In [16]:
# Cell 1: Imports & Setup
# Purpose: standard imports, plus device selection. Same as before, just
# restarting cleanly for the full encoder-decoder build.

import math
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [17]:
# Cell 2: Scaled Dot-Product Attention (Section 3.2.1)
# Purpose: the core attention computation --
#   Attention(Q, K, V) = softmax( Q K^T / sqrt(d_k) ) V
# Dividing by sqrt(d_k) counteracts large dot-product magnitudes that would
# otherwise push softmax into regions of very small gradients. The mask
# argument is what will let us implement BOTH padding masks (ignore
# meaningless positions) AND the decoder's causal mask (prevent attending
# to future positions) using the exact same function.

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q, K : (batch, heads, seq_q, d_k)
    V    : (batch, heads, seq_k, d_v)
    mask : broadcastable to (batch, heads, seq_q, seq_k); 0 = masked out
    Returns the attended values and the attention weight matrix.
    """
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)

    if mask is not None:
        scores = scores.masked_fill(mask == 0, float("-inf"))

    attn_weights = F.softmax(scores, dim=-1)
    return torch.matmul(attn_weights, V), attn_weights

In [18]:
# Cell 3: Multi-Head Attention (Section 3.2.2)
# Purpose: run several attention "heads" in parallel, each looking at a
# smaller subspace of the model dimension, then combine their outputs.
#   MultiHead(Q,K,V) = Concat(head_1, ..., head_h) W^O
#   head_i = Attention( Q W^Q_i, K W^K_i, V W^V_i )
# This same class will be reused for THREE different roles later: encoder
# self-attention, decoder masked self-attention, and decoder cross-attention
# -- the only thing that differs between those uses is what Q/K/V and mask
# get passed in, not the module itself.

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int):
        super().__init__()
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def _split_heads(self, x: torch.Tensor) -> torch.Tensor:
        """(batch, seq, d_model) -> (batch, heads, seq, d_k)"""
        B, S, _ = x.shape
        return x.view(B, S, self.num_heads, self.d_k).transpose(1, 2)

    def forward(self, Q, K, V, mask=None):
        B = Q.size(0)

        Q = self._split_heads(self.W_q(Q))
        K = self._split_heads(self.W_k(K))
        V = self._split_heads(self.W_v(V))

        x, _ = scaled_dot_product_attention(Q, K, V, mask)

        x = x.transpose(1, 2).contiguous().view(B, -1, self.num_heads * self.d_k)

        return self.W_o(x)

In [19]:
# Cell 4: Position-wise Feed-Forward Network (Section 3.3)
# Purpose: a small two-layer MLP applied independently to every position --
#   FFN(x) = max(0, x W_1 + b_1) W_2 + b_2
# Used identically inside both encoder layers and decoder layers.

class FeedForward(nn.Module):
    def __init__(self, d_model: int, d_ff: int):
        super().__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.linear2(F.relu(self.linear1(x)))

In [20]:
# Cell 5: Positional Encoding (Section 3.5)
# Purpose: inject position information into embeddings using fixed sin/cos
# waves of different frequencies per dimension --
#   PE(pos, 2i)   = sin( pos / 10000^(2i/d_model) )
#   PE(pos, 2i+1) = cos( pos / 10000^(2i/d_model) )
# Used identically for both the encoder's input embeddings and the
# decoder's input embeddings.

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512, dropout: float = 0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(0, d_model, 2, dtype=torch.float)
            * (-math.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        self.register_buffer("pe", pe.unsqueeze(0))   # (1, max_len, d_model)

    def forward(self, x):
        x = x + self.pe[:, : x.size(1)]
        return self.dropout(x)

In [21]:
# Cell 6: Encoder Layer + Encoder Stack (Section 3.1)
# Purpose: one encoder layer = self-attention sub-layer + feed-forward
# sub-layer, each wrapped in a residual connection and LayerNorm:
#   output = LayerNorm( x + Sublayer(x) )
# The encoder processes the SOURCE sequence (the sequence to be reversed)
# and produces a representation the decoder will later attend to via
# cross-attention.

class EncoderLayer(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = FeedForward(d_model, d_ff)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        # Sub-layer 1: self-attention (Q = K = V = x)
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, mask)))

        # Sub-layer 2: position-wise feed-forward
        x = self.norm2(x + self.dropout(self.feed_forward(x)))

        return x


class Encoder(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int,
                 num_layers: int, dropout: float = 0.1):
        super().__init__()
        self.layers = nn.ModuleList(
            [EncoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        )

    def forward(self, x, mask=None):
        for layer in self.layers:
            x = layer(x, mask)
        return x

In [22]:
# Cell 7: Decoder Layer + Decoder Stack (Section 3.1, right half of Figure 1)
# Purpose: unlike the encoder, each decoder layer has THREE sub-layers:
#   1. Masked self-attention (decoder attends to itself, but can't see
#      future positions -- this is what "masked" means, enforced via the
#      causal mask we'll build in Cell 8)
#   2. Cross-attention (Q comes from the decoder, K/V come from the
#      ENCODER's output -- this is how the decoder actually uses the
#      source sequence to decide what to generate)
#   3. Position-wise feed-forward (identical role to the encoder's)
# Same residual + LayerNorm pattern as the encoder, just one more sub-layer.

class DecoderLayer(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int, dropout: float = 0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads)
        self.cross_attn = MultiHeadAttention(d_model, num_heads)
        self.feed_forward = FeedForward(d_model, d_ff)

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, encoder_output, causal_mask):
        # Sub-layer 1: masked self-attention (Q=K=V=x, future positions blocked)
        x = self.norm1(x + self.dropout(self.self_attn(x, x, x, causal_mask)))

        # Sub-layer 2: cross-attention -- Q from decoder, K/V from encoder output.
        # No mask needed here: every decoder position is allowed to see the
        # ENTIRE source sequence, just not future TARGET positions.
        x = self.norm2(x + self.dropout(self.cross_attn(x, encoder_output, encoder_output)))

        # Sub-layer 3: position-wise feed-forward
        x = self.norm3(x + self.dropout(self.feed_forward(x)))

        return x


class Decoder(nn.Module):
    def __init__(self, d_model: int, num_heads: int, d_ff: int,
                 num_layers: int, dropout: float = 0.1):
        super().__init__()
        self.layers = nn.ModuleList(
            [DecoderLayer(d_model, num_heads, d_ff, dropout) for _ in range(num_layers)]
        )

    def forward(self, x, encoder_output, causal_mask):
        for layer in self.layers:
            x = layer(x, encoder_output, causal_mask)
        return x

In [23]:
# Cell 8: Causal Mask Generator (Section 3.2.3)
# Purpose: build the lower-triangular mask that prevents each decoder
# position from attending to any position AFTER itself. Position i is
# allowed to see positions 0..i, nothing beyond. This is what forces the
# decoder to generate autoregressively rather than "cheating" by looking
# ahead at the answer during training.

def generate_causal_mask(seq_len: int, device: torch.device):
    # torch.tril: lower-triangular part of a matrix of ones -> 1s on and
    # below the diagonal, 0s above it.
    mask = torch.tril(torch.ones(seq_len, seq_len, device=device))
    # Reshape to (1, 1, seq_len, seq_len) so it broadcasts across
    # (batch, heads, seq_len, seq_len) inside scaled_dot_product_attention.
    return mask.unsqueeze(0).unsqueeze(0)

In [24]:
# Cell 9: Full Seq2Seq Transformer
# Purpose: wire the encoder and decoder together. Separate embeddings for
# source and target (standard practice, even though they share a vocabulary
# here) since the encoder and decoder are learning different roles.

class Seq2SeqTransformer(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, num_heads: int,
                 d_ff: int, num_layers: int, max_len: int = 512, dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.src_embedding = nn.Embedding(vocab_size, d_model)
        self.tgt_embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = PositionalEncoding(d_model, max_len, dropout)
        self.encoder = Encoder(d_model, num_heads, d_ff, num_layers, dropout)
        self.decoder = Decoder(d_model, num_heads, d_ff, num_layers, dropout)
        self.output_proj = nn.Linear(d_model, vocab_size)

    def encode(self, src):
        x = self.src_embedding(src) * math.sqrt(self.d_model)
        x = self.pos_encoding(x)
        return self.encoder(x)

    def decode(self, tgt_in, encoder_output, causal_mask):
        x = self.tgt_embedding(tgt_in) * math.sqrt(self.d_model)
        x = self.pos_encoding(x)
        x = self.decoder(x, encoder_output, causal_mask)
        return self.output_proj(x)   # (batch, tgt_len, vocab_size)

    def forward(self, src, tgt_in, causal_mask):
        encoder_output = self.encode(src)
        return self.decode(tgt_in, encoder_output, causal_mask)

In [25]:
# Cell 10: Data Generator (with BOS/EOS for the decoder)
# Purpose: unlike the encoder-only version, the decoder needs special
# start-of-sequence (BOS) and end-of-sequence (EOS) tokens: BOS kicks off
# generation, EOS tells the model (and us, at inference time) when to stop.
# Token layout: 0=PAD (unused here, no padding needed at fixed length),
# 1=BOS, 2=EOS, content tokens are 3..vocab_size-1.
#
# Teacher forcing setup: during training, the decoder is fed the CORRECT
# target sequence shifted right by one position (tgt_in), and must predict
# the target sequence shifted left by one position (tgt_out). This lets us
# train in parallel across all positions at once, even though generation
# at inference time will be sequential (Cell 13).

PAD, BOS, EOS = 0, 1, 2
CONTENT_START = 3

def generate_batch(batch_size: int, seq_len: int, vocab_size: int, device: torch.device):
    src = torch.randint(CONTENT_START, vocab_size, (batch_size, seq_len), device=device)
    tgt = torch.flip(src, dims=[1])  # the reversed sequence -- the true task target

    bos_col = torch.full((batch_size, 1), BOS, dtype=torch.long, device=device)
    eos_col = torch.full((batch_size, 1), EOS, dtype=torch.long, device=device)

    tgt_full = torch.cat([tgt, eos_col], dim=1)          # (batch, seq_len+1): tgt then EOS
    tgt_in  = torch.cat([bos_col, tgt_full[:, :-1]], dim=1)  # decoder INPUT:  BOS, tgt[0..L-2]
    tgt_out = tgt_full                                    # decoder TARGET: tgt[0..L-1], EOS

    return src, tgt_in, tgt_out

In [26]:
# Cell 11: Hyperparameters, Model, Optimizer, LR Warmup Schedule (Section 5.3)
# Purpose: instantiate the full seq2seq model and set up the paper's actual
# learning-rate schedule -- warmup then decay -- since the earlier encoder-
# only experiment showed a fixed LR plateauing on a harder task.

VOCAB_SIZE   = 20     # PAD, BOS, EOS + 17 content tokens (3..19)
D_MODEL      = 64
NUM_HEADS    = 4
D_FF         = 128
NUM_LAYERS   = 2
SEQ_LEN      = 10
BATCH_SIZE   = 64
NUM_STEPS    = 8_000
WARMUP_STEPS = 400
DROPOUT      = 0.1

print(f"Model: d_model={D_MODEL}, heads={NUM_HEADS}, layers={NUM_LAYERS}, "
      f"d_ff={D_FF}, vocab={VOCAB_SIZE}, seq_len={SEQ_LEN}\n")

model = Seq2SeqTransformer(
    vocab_size=VOCAB_SIZE,
    d_model=D_MODEL,
    num_heads=NUM_HEADS,
    d_ff=D_FF,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
).to(device)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {total_params:,}\n")

optimizer = torch.optim.Adam(model.parameters(), lr=1.0, betas=(0.9, 0.98), eps=1e-9)
criterion = nn.CrossEntropyLoss(ignore_index=PAD)  # ignore_index unused here (no padding), kept for correctness

def lr_lambda(step):
    step = max(step, 1)
    return (D_MODEL ** -0.5) * min(step ** -0.5, step * WARMUP_STEPS ** -1.5)

scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

Model: d_model=64, heads=4, layers=2, d_ff=128, vocab=20, seq_len=10

Trainable parameters: 171,284



In [27]:
# Cell 12: Training Loop (teacher forcing)
# Purpose: train using teacher forcing -- feed the TRUE shifted target as
# decoder input (tgt_in) rather than the model's own previous predictions.
# This is standard for training seq2seq Transformers: it lets every decoder
# position be predicted in parallel (fast), at the cost of a train/inference
# mismatch that greedy decoding (Cell 13) will expose us to at test time.

model.train()
for step in range(1, NUM_STEPS + 1):
    src, tgt_in, tgt_out = generate_batch(BATCH_SIZE, SEQ_LEN, VOCAB_SIZE, device)

    causal_mask = generate_causal_mask(tgt_in.size(1), device)
    logits = model(src, tgt_in, causal_mask)   # (batch, tgt_len, vocab_size)

    loss = criterion(
        logits.reshape(-1, VOCAB_SIZE),
        tgt_out.reshape(-1),
    )

    optimizer.zero_grad()
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    scheduler.step()

    if step % 500 == 0 or step == 1:
        current_lr = scheduler.get_last_lr()[0]
        print(f"  step {step:5d}/{NUM_STEPS}  loss={loss.item():.4f}  lr={current_lr:.6f}")

print("\nTraining complete.")

  step     1/8000  loss=3.1850  lr=0.000016
  step   500/8000  loss=1.1771  lr=0.005590
  step  1000/8000  loss=0.5404  lr=0.003953
  step  1500/8000  loss=0.3245  lr=0.003227
  step  2000/8000  loss=0.1715  lr=0.002795
  step  2500/8000  loss=0.1117  lr=0.002500
  step  3000/8000  loss=0.0790  lr=0.002282
  step  3500/8000  loss=0.0938  lr=0.002113
  step  4000/8000  loss=0.0522  lr=0.001976
  step  4500/8000  loss=0.0560  lr=0.001863
  step  5000/8000  loss=0.0285  lr=0.001768
  step  5500/8000  loss=0.0298  lr=0.001685
  step  6000/8000  loss=0.0463  lr=0.001614
  step  6500/8000  loss=0.0122  lr=0.001550
  step  7000/8000  loss=0.0126  lr=0.001494
  step  7500/8000  loss=0.0101  lr=0.001443
  step  8000/8000  loss=0.0110  lr=0.001398

Training complete.


In [28]:
# Cell 13: Greedy Autoregressive Decoding
# Purpose: THIS is how a real Transformer generates output at inference
# time -- one token at a time, feeding each prediction back in as input for
# the next step, since (unlike training) we don't have the true answer to
# feed as tgt_in. Encoder runs once; decoder runs in a loop up to max_len
# steps or until EOS is generated.

def greedy_decode(model, src, max_len, device):
    model.eval()
    with torch.no_grad():
        encoder_output = model.encode(src)
        batch_size = src.size(0)

        tgt_in = torch.full((batch_size, 1), BOS, dtype=torch.long, device=device)

        for _ in range(max_len):
            causal_mask = generate_causal_mask(tgt_in.size(1), device)
            logits = model.decode(tgt_in, encoder_output, causal_mask)
            next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)  # last position's prediction
            tgt_in = torch.cat([tgt_in, next_token], dim=1)

    return tgt_in[:, 1:]  # drop the leading BOS before returning

In [29]:
# Cell 14: Evaluation
# Purpose: generate predictions via TRUE autoregressive decoding (not
# teacher forcing) and compare against the actual reversed target. Checks
# both exact-sequence accuracy and whether EOS was correctly placed.

def evaluate(model, device, vocab_size, seq_len, num_examples=8):
    src, tgt_in, tgt_out = generate_batch(num_examples, seq_len, vocab_size, device)
    predicted = greedy_decode(model, src, max_len=seq_len + 1, device=device)

    print("=" * 60)
    print("Reverse Task (Seq2Seq) -- Qualitative Examples")
    print("=" * 60)

    exact = 0
    for i in range(num_examples):
        inp = src[i].tolist()
        true_tgt = tgt_out[i].tolist()          # true reversed sequence + EOS
        pred = predicted[i].tolist()

        ok = pred == true_tgt
        exact += ok
        mark = "PASS" if ok else "FAIL"
        print(f"[{mark}]  input : {inp}")
        print(f"       target: {true_tgt}")
        print(f"       output: {pred}")
        print()

    print(f"Exact-sequence accuracy : {exact}/{num_examples}")

    # Token-level accuracy over a larger held-out sample
    src_big, _, tgt_out_big = generate_batch(500, seq_len, vocab_size, device)
    pred_big = greedy_decode(model, src_big, max_len=seq_len + 1, device=device)
    token_acc = (pred_big == tgt_out_big).float().mean().item()
    print(f"Token-level accuracy    : {token_acc * 100:.1f}%  (500 samples)")

evaluate(model, device, VOCAB_SIZE, SEQ_LEN)

Reverse Task (Seq2Seq) -- Qualitative Examples
[PASS]  input : [4, 17, 15, 9, 12, 17, 17, 6, 18, 7]
       target: [7, 18, 6, 17, 17, 12, 9, 15, 17, 4, 2]
       output: [7, 18, 6, 17, 17, 12, 9, 15, 17, 4, 2]

[PASS]  input : [15, 9, 13, 6, 7, 8, 13, 10, 18, 16]
       target: [16, 18, 10, 13, 8, 7, 6, 13, 9, 15, 2]
       output: [16, 18, 10, 13, 8, 7, 6, 13, 9, 15, 2]

[PASS]  input : [17, 3, 11, 16, 9, 3, 9, 11, 4, 8]
       target: [8, 4, 11, 9, 3, 9, 16, 11, 3, 17, 2]
       output: [8, 4, 11, 9, 3, 9, 16, 11, 3, 17, 2]

[PASS]  input : [6, 7, 14, 18, 15, 14, 7, 14, 14, 4]
       target: [4, 14, 14, 7, 14, 15, 18, 14, 7, 6, 2]
       output: [4, 14, 14, 7, 14, 15, 18, 14, 7, 6, 2]

[PASS]  input : [19, 9, 7, 5, 10, 7, 13, 17, 13, 12]
       target: [12, 13, 17, 13, 7, 10, 5, 7, 9, 19, 2]
       output: [12, 13, 17, 13, 7, 10, 5, 7, 9, 19, 2]

[PASS]  input : [13, 9, 7, 9, 13, 9, 15, 3, 19, 19]
       target: [19, 19, 3, 15, 9, 13, 9, 7, 9, 13, 2]
       output: [19, 19, 3, 15, 9,